# Point-Supervised Semantic Segmentation on LoveDA

This notebook walks through the whole pipeline:

1. the partial focal loss (and its equivalence to partial cross-entropy when `gamma=0`),
2. point-annotation simulation on a real LoveDA crop,
3. the full-mask isolation guarantee of the training dataset,
4. the U-Net model and a tiny end-to-end training run,
5. the annotation-density and sampling-strategy experiment results.

It imports the `pointseg` package, so the notebook and the CLI run exactly the same code.
Run it from the repository root or from `notebooks/`; paths are resolved automatically.


In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    # Walk upwards until the repository root (src/pointseg + configs) is found.
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "pointseg").is_dir() and (
            candidate / "configs" / "default.yaml"
        ).is_file():
            return candidate
    raise RuntimeError(f"could not locate the pointseg repository from {start}")


ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import matplotlib.pyplot as plt
import numpy as np
import torch

from pointseg.config import load_config
from pointseg.constants import CLASS_NAMES, IGNORE_INDEX
from pointseg.datasets.loveda import discover_pairs
from pointseg.datasets.point_sampling import (
    sample_class_balanced_points,
    sample_random_points,
)
from pointseg.datasets.transforms import remap_mask
from pointseg.losses import PartialCrossEntropyLoss, PartialFocalLoss
from pointseg.models.unet import build_model, describe_model

print("project root:", ROOT)
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Partial focal loss

The assignment defines

$$L_{pFCE} = \frac{\sum_i \mathrm{Focal}(pred_i, GT_i)\, MASK_{labeled,i}}{\sum_i MASK_{labeled,i}}$$

with $FL = -\alpha_t (1-p_t)^\gamma \log p_t$. Only labelled pixels enter the numerator
and the denominator. With `gamma=0` and no class weights this is exactly partial CE.


In [ ]:
logits = torch.tensor([[[[2.0, -1.0, 0.5]], [[0.1, 1.5, -0.5]]]])
target = torch.tensor([[[0, 1, IGNORE_INDEX]]])

focal = PartialFocalLoss(gamma=2.0)
ce = PartialCrossEntropyLoss()

print(f"partial focal (gamma=2): {focal(logits, target).item():.6f}")
print(f"partial CE:              {ce(logits, target).item():.6f}")
print(f"focal gamma=0:           {PartialFocalLoss(gamma=0.0)(logits, target).item():.6f}")
print(
    "gamma=0 equals partial CE:",
    torch.allclose(focal.__class__(gamma=0.0)(logits, target), ce(logits, target), atol=1e-6),
)

## 2. Point annotation simulation

A fully labelled crop is reduced to N labelled pixels. The random strategy follows the
natural class frequency; the class-balanced strategy round-robins over the classes
present so rare classes are not drowned out.


In [ ]:
import cv2

pairs = discover_pairs("data/LoveDA", "Train")


def read_pair(pair):
    image = cv2.cvtColor(cv2.imread(str(pair["image"]), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    mask = remap_mask(cv2.imread(str(pair["mask"]), cv2.IMREAD_UNCHANGED))
    return image, mask


# Pick a crop that contains several classes, so the two sampling strategies
# actually differ in the figure below.
best = None
for pair in pairs[:25]:
    candidate_image, candidate_mask = read_pair(pair)
    centre = candidate_mask[256:768, 256:768]
    n_classes = len(np.unique(centre[centre != IGNORE_INDEX]))
    if best is None or n_classes > best[0]:
        best = (n_classes, candidate_image, centre, pair["id"])
n_classes, image_crop, mask_crop, sample_id = best
print(f"using {sample_id}: {n_classes} classes in the centre 512x512 crop")

random_points = sample_random_points(mask_crop, 100, seed=0)
balanced_points = sample_class_balanced_points(mask_crop, 100, seed=0)


from pointseg.evaluation.visualize import colorize, points_overlay

dimmed = (image_crop * 0.55).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
axes[0].imshow(image_crop)
axes[0].set_title("image")
axes[1].imshow(colorize(mask_crop))
axes[1].set_title("full mask")
axes[2].imshow(points_overlay(random_points, base=dimmed))
axes[2].set_title("100 random points")
axes[3].imshow(points_overlay(balanced_points, base=dimmed))
axes[3].set_title("100 class-balanced points")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(
    "labelled pixels:",
    int((random_points != IGNORE_INDEX).sum()),
    "| classes:",
    np.unique(random_points[random_points != IGNORE_INDEX]),
)
print(
    "class counts (balanced):",
    {CLASS_NAMES[c]: int((balanced_points == c).sum()) for c in range(7)},
)

## 3. Full-mask isolation

The training dataset samples points inside `__getitem__` and returns only the sparse mask.
The full mask is deleted before the item leaves the method, so it cannot leak into the loss.


In [ ]:
from pointseg.datasets.loveda import LoveDATrainDataset

train_ds = LoveDATrainDataset(
    "data/LoveDA",
    crop_size=512,
    points_per_crop=100,
    strategy="random",
    fixed_points=True,
    seed=42,
    max_images=50,
    augment_data=False,
)
item = train_ds[0]
labelled = int((item["mask"] != IGNORE_INDEX).sum())
print("returned keys:", sorted(item))
print("sparse mask shape:", tuple(item["mask"].shape), "| labelled pixels:", labelled)
assert labelled <= 100
assert set(item["mask"][item["mask"] != IGNORE_INDEX].unique().tolist()) <= set(range(7))

train_ds.set_epoch(1)
second = train_ds[0]["mask"]
print("fixed point set is identical across epochs:", torch.equal(item["mask"], second))

## 4. Model

U-Net with a ResNet-34 encoder, ImageNet weights, raw logits for seven classes.


In [ ]:
config = load_config("configs/default.yaml")
model = build_model(config.model).eval()
print(describe_model(model))
with torch.no_grad():
    out = model(torch.randn(1, 3, 512, 512))
print("output shape:", tuple(out.shape), "(raw logits)")

## 5. Tiny end-to-end training run

The full experiments use `pointseg-train` / `pointseg-exp-density` on the GPU.
Here we run the same `train()` function for one epoch on eight images so the notebook
stays fast; artefacts are written to `outputs/notebook_demo/`.


In [ ]:
from pointseg.training.train import train

demo = load_config("configs/smoke.yaml")
demo.dataset.root = "data/LoveDA"
demo.dataset.max_train_images = 8
demo.dataset.max_val_images = 4
demo.training.epochs = 1
demo.training.max_train_batches = 2
demo.training.num_workers = 0
demo.training.amp = False
demo.model.pretrained = False
demo.evaluation.tile_size = 128
demo.evaluation.tile_overlap = 0.0
demo.evaluation.batch_size = 1
demo.output_dir = "outputs/notebook_demo"
demo.run_name = "demo"
summary = train(demo, device=torch.device("cpu"))
print("demo best mIoU:", round(summary["best_miou"], 4))

## 6. Experiment results

Every number below is read from a real `metrics.json` produced by the experiment runners.
If the experiments have not been run yet, the cell reports that instead of inventing values.


In [ ]:
import json
from pathlib import Path

rows = []
for path in sorted(Path("outputs").rglob("metrics.json")):
    # Only the experiment runners count; ad-hoc smoke/benchmark runs are excluded.
    if not {"exp_density", "exp_sampling"} & set(path.parts):
        continue
    data = json.loads(path.read_text())
    cfg = data.get("config", {})
    ps = cfg.get("point_sampling", {})
    rows.append(
        {
            "run": data.get("run_name", path.parent.name),
            "strategy": ps.get("strategy"),
            "points": ps.get("points_per_crop"),
            "best_mIoU": data.get("best_miou"),
            "mean_F1": (data.get("best_metrics") or {}).get("mean_f1"),
            "pixel_acc": (data.get("best_metrics") or {}).get("pixel_accuracy"),
        }
    )

if rows:
    rows.sort(key=lambda row: (str(row["strategy"]), int(row["points"] or 0)))
    header = (
        f"{'run':<38} {'strategy':<15} {'points':>6} {'mIoU':>7} {'mean F1':>8} {'pixel acc':>9}"
    )
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['run']:<38} {str(row['strategy']):<15} {int(row['points']):>6} "
            f"{row['best_mIoU']:>7.4f} {row['mean_F1']:>8.4f} {row['pixel_acc']:>9.4f}"
        )
else:
    print("No experiment outputs found. Run pointseg-exp-density / pointseg-exp-sampling first.")

### Training curves and per-class metrics

In [ ]:
from IPython.display import Image, display

for candidate in [
    "outputs/exp_density/density_100pts_seed42/figures/history.png",
    "outputs/exp_density/density_100pts_seed42/figures/per_class.png",
    "outputs/exp_density/density_100pts_seed42/figures/predictions.png",
]:
    if Path(candidate).is_file():
        display(Image(filename=candidate))
    else:
        print("missing (run the experiments first):", candidate)

## 7. Takeaways

- The partial focal loss only ever sees labelled pixels; `gamma=0` reproduces partial CE.
- A fixed point set keeps the annotation budget honest across epochs.
- More points generally improve mIoU, with diminishing returns; class-balanced sampling
  changes which classes benefit rather than uniformly improving the mean.
- Validation always uses full ground-truth masks, so the numbers reflect real segmentation
  quality rather than the sparse training signal.

Limitations: a single seed and a 600/200-image subset on one T4, and no semi-supervised
use of unlabelled pixels (see the technical report for the full discussion).
